# Vector Calculus Product Rules (iv) and (v) with Plane-Wave Test Fields

This notebook proves/checks two Griffiths product rules:

**(iv)**
$$\nabla\cdot(\mathbf A\times\mathbf B)=\mathbf B\cdot(\nabla\times\mathbf A)-\mathbf A\cdot(\nabla\times\mathbf B)$$

**(v)**
$$\nabla\times(f\mathbf A)=f(\nabla\times\mathbf A)-\mathbf A\times(\nabla f)$$

Then it tests them on complex plane-wave fields.

In [ ]:
import sympy as sp

x, y, z = sp.symbols('x y z', real=True)
I = sp.I

def grad(f):
    return sp.Matrix([sp.diff(f, x), sp.diff(f, y), sp.diff(f, z)])

def div(A):
    return sp.diff(A[0], x) + sp.diff(A[1], y) + sp.diff(A[2], z)

def curl(A):
    return sp.Matrix([
        sp.diff(A[2], y) - sp.diff(A[1], z),
        sp.diff(A[0], z) - sp.diff(A[2], x),
        sp.diff(A[1], x) - sp.diff(A[0], y)
    ])

def dot(A, B):
    return (A.T * B)[0]

def cross(A, B):
    return sp.Matrix([
        A[1]*B[2] - A[2]*B[1],
        A[2]*B[0] - A[0]*B[2],
        A[0]*B[1] - A[1]*B[0]
    ])

def zero_simplify(expr):
    return sp.simplify(sp.expand(expr))

## 1. Symbolic proof of rule (iv)

Use arbitrary component functions:

$$\mathbf A=(A_x,A_y,A_z),\qquad \mathbf B=(B_x,B_y,B_z).$$

In [ ]:
Ax, Ay, Az = [sp.Function(name)(x, y, z) for name in ['Ax', 'Ay', 'Az']]
Bx, By, Bz = [sp.Function(name)(x, y, z) for name in ['Bx', 'By', 'Bz']]

A = sp.Matrix([Ax, Ay, Az])
B = sp.Matrix([Bx, By, Bz])

lhs_iv = div(cross(A, B))
rhs_iv = dot(B, curl(A)) - dot(A, curl(B))

sp.simplify(lhs_iv - rhs_iv)

If the output is `0`, SymPy has verified:

$$\nabla\cdot(\mathbf A\times\mathbf B)=\mathbf B\cdot(\nabla\times\mathbf A)-\mathbf A\cdot(\nabla\times\mathbf B).$$

## 2. Symbolic proof of rule (v)

Use an arbitrary scalar field:

$$f=f(x,y,z).$$

In [ ]:
f = sp.Function('f')(x, y, z)

lhs_v = curl(f*A)
rhs_v = f*curl(A) - cross(A, grad(f))

sp.simplify(lhs_v - rhs_v)

If the output is the zero vector, SymPy has verified:

$$\nabla\times(f\mathbf A)=f(\nabla\times\mathbf A)-\mathbf A\times(\nabla f).$$

## 3. Plane-wave physics test

Let

$$\mathbf A=\mathbf a e^{i\mathbf k\cdot\mathbf r},\qquad \mathbf B=\mathbf b e^{i\mathbf q\cdot\mathbf r},\qquad f=e^{i\mathbf p\cdot\mathbf r}.$$

For a plane wave,

$$\nabla e^{i\mathbf k\cdot\mathbf r}=i\mathbf k e^{i\mathbf k\cdot\mathbf r}.$$

So `grad`, `curl`, and `div` become algebraic operations involving the wave vector.

In [ ]:
kx, ky, kz, qx, qy, qz, px, py, pz = sp.symbols('kx ky kz qx qy qz px py pz', real=True)
a1, a2, a3, b1, b2, b3 = sp.symbols('a1 a2 a3 b1 b2 b3')

r_k = kx*x + ky*y + kz*z
r_q = qx*x + qy*y + qz*z
r_p = px*x + py*y + pz*z

A_pw = sp.Matrix([a1, a2, a3]) * sp.exp(I*r_k)
B_pw = sp.Matrix([b1, b2, b3]) * sp.exp(I*r_q)
f_pw = sp.exp(I*r_p)

test_iv = sp.simplify(div(cross(A_pw, B_pw)) - (dot(B_pw, curl(A_pw)) - dot(A_pw, curl(B_pw))))
test_v = sp.simplify(curl(f_pw*A_pw) - (f_pw*curl(A_pw) - cross(A_pw, grad(f_pw))))

test_iv, test_v

Expected output:

```python
(0, Matrix([[0], [0], [0]]))
```

That means both rules work for complex plane waves.

## 4. Numerical demo with actual values

Now choose specific wave vectors and polarizations.

In [ ]:
subs_values = {
    kx: 1, ky: 2, kz: -1,
    qx: -2, qy: 1, qz: 3,
    px: 1, py: -1, pz: 2,
    a1: 1, a2: 2, a3: -1,
    b1: 0, b2: 3, b3: 1,
    x: 0.25, y: -0.5, z: 0.75
}

print('Rule (iv) difference =', complex(sp.N(test_iv.subs(subs_values))))
print('Rule (v) difference =')
for component in test_v:
    print(complex(sp.N(component.subs(subs_values))))

## 5. Physics interpretation

- Rule **(iv)** says the divergence of a cross-product field is controlled by the curls of the original fields.
- Rule **(v)** says when a vector field is multiplied by a scalar wave, its curl gets two pieces:
  - curl of the original vector field,
  - plus the extra rotation caused by the scalar field changing in space.

For plane waves, derivatives turn into multiplication by `i * wave_vector`. That is why Fourier methods are powerful in electromagnetism, quantum mechanics, and signal processing.